# Transfer learning + ONNX + kwantyzacja INT8 — Alfabet języka migowego (ASL)

Notebook realizuje punkty **1–7** projektu:
1. Gotowy model (MobileNetV2, pretrenowany na ImageNet)
2. Transfer learning na zbiorze **ASL Alphabet** (Kaggle) — 29 klas, których ImageNet nie zna
3. Eksport do **ONNX**
4. Sprawdzenie zgodności predykcji PyTorch vs ONNX Runtime
5. Pomiar czasu inferencji (PyTorch CPU, ONNX Runtime CPU)
6. Kwantyzacja do **ONNX INT8**
7. Pomiar czasu modelu skwantyzowanego + porównanie

> **Środowisko wykonawcze:** ustaw w Colabie *Runtime → Change runtime type → GPU* (do treningu).
> Inferencję i pomiary robimy świadomie na **CPU**, bo o to chodzi w projekcie.

> **Zamiana na ResNet18:** w komórce z modelem zakomentuj MobileNetV2 i odkomentuj wariant ResNet18 — reszta notebooka działa bez zmian.

> ⚠️ **WAŻNE — uruchamiaj przez `Runtime → Restart session and run all`.** Notebook ma stan globalny; ponowne odpalenie komórki z modelem po treningu tworzy świeżą, nieprzetrenowaną głowę i eksportuje zły model. Przebieg z góry na dół, raz.

In [ ]:
# Instalacja zaleznosci (Colab ma juz torch + torchvision z CUDA)
!pip install -q onnx onnxruntime kaggle pandas

In [ ]:
import os, time, random, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urzadzenie treningowe:", device)

## 1–2. Pobranie zbioru danych z Kaggle

Jak zdobyć `kaggle.json`: wejdź na Kaggle → ikona profilu → **Settings** → sekcja **API** → **Create New Token**.
Pobierze się plik `kaggle.json`. Uruchom komórkę poniżej i wgraj go, gdy pojawi się okno wyboru pliku.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !unzip -q -o "/content/asl-alphabet.zip" -d /content/data
!unzip -q -o "/content/drive/MyDrive/Colab Notebooks/datasets/asl-alphabet.zip" -d /content/data

print("Pobrano i rozpakowano.")

In [ ]:
# Konfiguracja
TRAIN_DIR = "data/asl_alphabet_train/asl_alphabet_train"  # jesli sciezka inna, popraw
print("Przyklad klas:", sorted(os.listdir(TRAIN_DIR))[:8], "...")
print("Liczba klas:", len(os.listdir(TRAIN_DIR)))

N_PER_CLASS = 400   # podzbior na klase - zmniejsz, jesli trening za wolny
IMG_SIZE = 224
BATCH = 64

In [ ]:
mean = [0.485, 0.456, 0.406]   # statystyki ImageNet - MUSZA byc identyczne w aplikacji!
std  = [0.229, 0.224, 0.225]

# UWAGA: dla jezyka migowego NIE odbijamy obrazu w poziomie - zmienieni to znaczenie gestu.
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Dwie instancje ImageFolder na ten sam katalog: jedna z augmentacja (trening), druga bez (walidacja).
train_full = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
eval_full  = datasets.ImageFolder(TRAIN_DIR, transform=eval_tf)
class_names = train_full.classes
num_classes = len(class_names)

# Podzbior N obrazow na klase + podzial 80/20 na train/val
from collections import defaultdict
by_class = defaultdict(list)
for idx, (_, label) in enumerate(train_full.samples):
    by_class[label].append(idx)

train_idx, val_idx = [], []
for label, idxs in by_class.items():
    chosen = random.sample(idxs, min(N_PER_CLASS, len(idxs)))
    split = int(0.8 * len(chosen))
    train_idx += chosen[:split]
    val_idx   += chosen[split:]

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(eval_full,  val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

print(f"Treningowe: {len(train_ds)}, walidacyjne: {len(val_ds)}, klasy: {num_classes}")

In [ ]:
# --- MODEL: MobileNetV2 (domyslny) ---
weights = MobileNet_V2_Weights.IMAGENET1K_V1
model = mobilenet_v2(weights=weights)
for p in model.features.parameters():     # zamrazamy backbone
    p.requires_grad = False
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)   # nowa glowa

# --- ALTERNATYWA: ResNet18 (odkomentuj caly blok, zakomentuj powyzszy) ---
# from torchvision.models import resnet18, ResNet18_Weights
# model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# for p in model.parameters():
#     p.requires_grad = False
# model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)
trainable = [p for p in model.parameters() if p.requires_grad]
print("Trenowalnych tensorow:", len(trainable))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(trainable, lr=1e-3)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    print(f"Epoka {epoch+1}/{EPOCHS}  loss={running/len(train_ds):.4f}")

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            pred = model(imgs).argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total

acc_torch = accuracy(model, val_loader)
print(f"Dokladnosc (val): {acc_torch*100:.2f}%")
# Jesli za niska: zwieksz N_PER_CLASS/EPOCHS lub odmroz wiecej warstw backbone'u.

In [ ]:
# Zapis wag PyTorch + nazw klas (potrzebne pozniej w aplikacji)
model_cpu = model.to("cpu").eval()
torch.save(model_cpu.state_dict(), "mobilenetv2_asl.pth")
with open("class_names.json", "w") as f:
    json.dump(class_names, f, ensure_ascii=False)
print("Zapisano wagi i nazwy klas.")

## 3. Eksport do ONNX

In [ ]:
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model_cpu, dummy, "mobilenetv2_asl.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},  # zmienny rozmiar batcha
    opset_version=13,
    dynamo=False,
)
print("Wyeksportowano do ONNX.")

## 4. Zgodnosc predykcji PyTorch vs ONNX Runtime

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession("mobilenetv2_asl.onnx", providers=["CPUExecutionProvider"])

x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    torch_out = model_cpu(x).numpy()
onnx_out = sess.run(None, {"input": x.numpy()})[0]

np.testing.assert_allclose(torch_out, onnx_out, rtol=1e-3, atol=1e-5)
print("OK - predykcje PyTorch i ONNX zgodne.")
print("Maks. roznica logitow:", float(np.abs(torch_out - onnx_out).max()))
print("Ten sam argmax:", int(torch_out.argmax()) == int(onnx_out.argmax()))

## 5. Pomiar czasu: PyTorch CPU vs ONNX Runtime CPU

Pomiar z rozgrzewka i usrednianiem — inaczej liczby sa losowe.

In [ ]:
def benchmark(fn, x, n=100, warmup=10):
    for _ in range(warmup):
        fn(x)
    t0 = time.perf_counter()
    for _ in range(n):
        fn(x)
    return (time.perf_counter() - t0) / n * 1000.0   # ms na inferencje

x_np = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)

# PyTorch na CPU
def torch_infer(x):
    with torch.no_grad():
        return model_cpu(torch.from_numpy(x))
t_torch = benchmark(torch_infer, x_np)

# ONNX Runtime na CPU
def onnx_infer(x):
    return sess.run(None, {"input": x})
t_onnx = benchmark(onnx_infer, x_np)

print(f"PyTorch CPU:      {t_torch:.2f} ms")
print(f"ONNX Runtime CPU: {t_onnx:.2f} ms")

## 6. Kwantyzacja do ONNX INT8 (statyczna)

Dla klasyfikatorow obrazow stosujemy kwantyzacje **statyczna** (`quantize_static`): liczy skale aktywacji na malym zbiorze kalibracyjnym i uzywa formatu QDQ. Daje realnie mniejszy model i jest wlasciwa metoda dla sieci konwolucyjnych.

> Uwaga: MobileNetV2 opiera sie na konwolucjach *depthwise*, dla ktorych ONNX Runtime na CPU nie ma wydajnych jader INT8 — wiec model bedzie ~4x mniejszy, ale na CPU moze nie byc szybszy. To dobra obserwacja do raportu. Dla pewnego przyspieszenia INT8 przelacz model na ResNet18 (komorka z modelem).

In [ ]:
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantType, QuantFormat
)

# Reader kalibracyjny: kilka batchy z walidacji jako pojedyncze probki (batch=1)
class ASLCalibrationReader(CalibrationDataReader):
    def __init__(self, loader, n_batches=4):
        self.data = []
        for i, (imgs, _) in enumerate(loader):
            if i >= n_batches:
                break
            for img in imgs:
                self.data.append({"input": img.unsqueeze(0).numpy().astype(np.float32)})
        self.it = iter(self.data)
    def get_next(self):
        return next(self.it, None)

quantize_static(
    "mobilenetv2_asl.onnx",
    "mobilenetv2_asl_int8.onnx",
    ASLCalibrationReader(val_loader),
    quant_format=QuantFormat.QDQ,
    per_channel=True,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QInt8,
)
print("Zapisano model INT8 (kwantyzacja statyczna).")

## 7. Pomiar modelu INT8 + porownanie wszystkich wariantow

In [ ]:
sess_int8 = ort.InferenceSession("mobilenetv2_asl_int8.onnx", providers=["CPUExecutionProvider"])

def onnx_int8_infer(x):
    return sess_int8.run(None, {"input": x})
t_int8 = benchmark(onnx_int8_infer, x_np)
print(f"ONNX INT8 CPU: {t_int8:.2f} ms")

In [ ]:
# Dokladnosc na CALYM zbiorze walidacyjnym (FP32 powinno sie zgadzac z komorka 10)
def onnx_accuracy(session, loader):
    correct = total = 0
    for imgs, labels in loader:
        out = session.run(None, {"input": imgs.numpy().astype(np.float32)})[0]
        correct += int((out.argmax(1) == labels.numpy()).sum())
        total += labels.size(0)
    return correct / total

acc_fp32 = onnx_accuracy(sess, val_loader)
acc_int8 = onnx_accuracy(sess_int8, val_loader)
print(f"Dokladnosc ONNX FP32: {acc_fp32*100:.2f}%")
print(f"Dokladnosc ONNX INT8: {acc_int8*100:.2f}%")

In [ ]:
import pandas as pd
def size_mb(p): return round(os.path.getsize(p) / 1e6, 2)

df = pd.DataFrame([
    {"Wariant": "PyTorch CPU",      "Czas [ms]": round(t_torch, 2), "Rozmiar [MB]": size_mb("mobilenetv2_asl.pth"),     "Dokladnosc [%]": round(acc_torch*100, 2)},
    {"Wariant": "ONNX Runtime CPU", "Czas [ms]": round(t_onnx, 2),  "Rozmiar [MB]": size_mb("mobilenetv2_asl.onnx"),    "Dokladnosc [%]": round(acc_fp32*100, 2)},
    {"Wariant": "ONNX INT8 CPU",    "Czas [ms]": round(t_int8, 2),  "Rozmiar [MB]": size_mb("mobilenetv2_asl_int8.onnx"), "Dokladnosc [%]": round(acc_int8*100, 2)},
])
df["Przyspieszenie vs PyTorch"] = (t_torch / df["Czas [ms]"]).round(2)
df

## Pobranie artefaktow do aplikacji FastAPI

Te trzy pliki przeniesiesz do projektu z backendem (aplikacja wczytuje **model INT8**).

In [ ]:
from google.colab import files
for f in ["mobilenetv2_asl_int8.onnx", "mobilenetv2_asl.onnx", "class_names.json"]:
    files.download(f)